In [45]:
#pip install pandas

In [140]:
from astroquery.alma import Alma
import os
import numpy as np
import pandas as pd
from astroquery.alma.utils import parse_frequency_support
import astropy.units as u
from astropy.table import Table
import shutil
from pathlib import Path
import tarfile

# These are the same keywords on the ALMA Archive: 1) Disks around low-mass stars, 2) Low mass star formation, 3) Outflows, jets, and ionized winds

In [25]:
# Give me all columns from the database. We are searching rowns with ALMA metadata
alma = Alma()
#query = """
#SELECT *
#FROM ivoa.obscore
#WHERE science_keyword LIKE '%Outflows, jets and ionized winds%'
#"""

In [102]:
#Used ALMA astroTAP query to find the data products with science keywords
query = """
SELECT
    obs_publisher_did,
    member_ous_uid,
    target_name,
    frequency_support,
    science_keyword
FROM ivoa.obscore
WHERE dataproduct_type = 'cube'
AND (
      science_keyword LIKE '%outflow%'
      OR science_keyword LIKE '%jet%'
      OR science_keyword LIKE '%wind%'
  )
"""

In [103]:
#This results have all the data products
results = alma.query_tap(query)

In [104]:

print(results)

<DALResultsTable length=41030>
     obs_publisher_did      ...
                            ...
           str33            ...
--------------------------- ...
ADS/JAO.ALMA#2017.1.01053.S ...
ADS/JAO.ALMA#2017.1.01053.S ...
ADS/JAO.ALMA#2017.1.01053.S ...
ADS/JAO.ALMA#2017.1.01053.S ...
ADS/JAO.ALMA#2025.1.00414.S ...
ADS/JAO.ALMA#2025.1.00414.S ...
ADS/JAO.ALMA#2025.1.00414.S ...
ADS/JAO.ALMA#2025.1.00414.S ...
ADS/JAO.ALMA#2024.1.01427.S ...
                        ... ...
ADS/JAO.ALMA#2023.1.01086.V ...
ADS/JAO.ALMA#2023.1.01086.V ...
ADS/JAO.ALMA#2023.1.01086.V ...
ADS/JAO.ALMA#2021.1.00120.S ...
ADS/JAO.ALMA#2015.1.00662.S ...
ADS/JAO.ALMA#2015.1.00662.S ...
ADS/JAO.ALMA#2015.1.00662.S ...
ADS/JAO.ALMA#2015.1.00662.S ...
ADS/JAO.ALMA#2021.1.00120.S ...


In [105]:
#Freuency support stored as string so need to parse frequncy and only need frequncy ranges
frequency_support_str = results['frequency_support'][0]
print(frequency_support_str)

[300.16..302.16GHz,1128.91kHz,8.1mJy/beam@10km/s,577.3uJy/beam@native, XX YY] U [302.03..304.03GHz,1128.91kHz,7.5mJy/beam@10km/s,534.9uJy/beam@native, XX YY] U [312.14..314.14GHz,1128.91kHz,8.1mJy/beam@10km/s,586.3uJy/beam@native, XX YY] U [314.01..316.01GHz,1128.91kHz,9.1mJy/beam@10km/s,658.8uJy/beam@native, XX YY]


In [106]:
freq_ranges = parse_frequency_support(frequency_support_str)
print(freq_ranges)

[[300.16 302.16]
 [302.03 304.03]
 [312.14 314.14]
 [314.01 316.01]] GHz


In [107]:
min_window = 346.5 * u.GHz
max_window = 347.8 * u.GHz

matches = [] #Empty list

for x in results:
    ranges = parse_frequency_support(x['frequency_support'])

    for fmin, fmax in ranges:
        if fmax >= min_window and fmin <= max_window:
            matches.append(x)
            break


In [108]:
table = results.to_table()

In [109]:
table.colnames

['obs_publisher_did',
 'member_ous_uid',
 'target_name',
 'frequency_support',
 'science_keyword']

In [110]:

matches = Table(rows=matches, names=table.colnames)
print(len(matches))

1503


In [111]:
print(matches[0])

     obs_publisher_did          member_ous_uid    target_name                                                                                                                                                       frequency_support                                                                                                                                                                       science_keyword                
--------------------------- --------------------- ----------- ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- -----------------------------------------------
ADS/JAO.ALMA#2017.1.01053.S uid://A001/X1288/Xf99       CG_30 [345.23..347.23GHz,1128.91kHz,8.3mJy/beam@10km/s,627.3uJy/beam@native, XX YY] U [3

In [115]:
uid = matches['member_ous_uid'][0]
print(uid)

uid://A001/X1288/Xf99


In [130]:
products = alma.get_data_info(uid)

print(products)

          ID          ... link_authorized
                      ...                
--------------------- ... ---------------
uid://A001/X1288/Xf99 ...            True
uid://A001/X1288/Xf99 ...            True
uid://A001/X1288/Xf99 ...              --
uid://A001/X1288/Xf99 ...            True
uid://A001/X1288/Xf99 ...              --
uid://A001/X1288/Xf99 ...            True


In [145]:
for row in products:
    print(row['access_url'])
    print(len(row))

https://almascience.nrao.edu/dataPortal/member.uid___A001_X1288_Xf99.README.txt
12
https://almascience.nrao.edu/dataPortal/2017.1.01053.S_uid___A001_X1288_Xf99_001_of_001.tar
12

12
https://almascience.nrao.edu/dataPortal/2017.1.01053.S_uid___A001_X1288_Xf99_auxiliary.tar
12

12
https://almascience.nrao.edu/dataPortal/2017.1.01053.S_uid___A002_Xc8cb70_X358f.asdm.sdm.tar
12


In [148]:
mask = []

for url in products['access_url']:
    url = str(url)

    if "_of_" in url:
        mask.append(True)
    else:
        mask.append(False)

science_tars = products[mask]

In [149]:
urls = science_tars['access_url'].astype(str)


In [150]:
alma.cache_location = '/users/skaur/Machine-Learning-Project/Astromorph_project/alma_astroquery'
download_dir = Path("alma_TAP")
download_dir.mkdir(exist_ok=True)

downloaded_files = alma.download_files(urls)

for f in downloaded_files:
    shutil.move(f, download_dir / Path(f).name)

In [142]:
# Open a .tar files
for tar_path in download_dir.glob("*.tar"):
    extract_dir = download_dir / tar_path.stem
    extract_dir.mkdir(exist_ok=True)

    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(path=extract_dir)

    print("Extracted:", tar_path.name)

/var/folders/rv/2r24b49j7kd29j3gsm48r3c40000gp/T/ipykernel_45446/3593851651.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extracted: 2017.1.01053.S_uid___A001_X1288_Xf99_001_of_001.tar


In [143]:
#Only extract pb and pbcor.fits files
fits_files = []

for f in download_dir.rglob("*.fits"):
    name = f.name.lower()

    if "pbcor" in name or "_pb.fits" in name:
        fits_files.append(f)

print("PB/PBCOR files found:", len(fits_files))

for f in fits_files[:10]:
    print(f)

PB/PBCOR files found: 17
alma_TAP/2017.1.01053.S_uid___A001_X1288_Xf99_001_of_001/2017.1.01053.S/science_goal.uid___A001_X1288_Xf91/group.uid___A001_X1288_Xf98/member.uid___A001_X1288_Xf99/product/member.uid___A001_X1288_Xf99.J0522-3627_bp.spw22.mfs.I.pbcor.fits
alma_TAP/2017.1.01053.S_uid___A001_X1288_Xf99_001_of_001/2017.1.01053.S/science_goal.uid___A001_X1288_Xf91/group.uid___A001_X1288_Xf98/member.uid___A001_X1288_Xf99/product/member.uid___A001_X1288_Xf99.CG_30_sci.spw22.mfs.I.pbcor.fits
alma_TAP/2017.1.01053.S_uid___A001_X1288_Xf99_001_of_001/2017.1.01053.S/science_goal.uid___A001_X1288_Xf91/group.uid___A001_X1288_Xf98/member.uid___A001_X1288_Xf99/product/member.uid___A001_X1288_Xf99.J0522-3627_bp.spw18.mfs.I.pbcor.fits
alma_TAP/2017.1.01053.S_uid___A001_X1288_Xf99_001_of_001/2017.1.01053.S/science_goal.uid___A001_X1288_Xf91/group.uid___A001_X1288_Xf98/member.uid___A001_X1288_Xf99/product/member.uid___A001_X1288_Xf99.CG_30_sci.spw18.mfs.I.pbcor.fits
alma_TAP/2017.1.01053.S_uid___A

In [151]:
#Alma.help_tap()